In [50]:
import pandas as pd

# Leer datasets
df1 = pd.read_csv("DataScience_salaries_2024.csv")
df2 = pd.read_csv("ds_salaries.csv", index_col=0)
df3 = pd.read_csv("Latest_Data_Science_Salaries.csv")

In [51]:
df1.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2021,MI,FT,Data Scientist,30400000,CLP,40038,CL,100,CL,L
1,2021,MI,FT,BI Data Analyst,11000000,HUF,36259,HU,50,US,L
2,2020,MI,FT,Data Scientist,11000000,HUF,35735,HU,50,HU,L
3,2021,MI,FT,ML Engineer,8500000,JPY,77364,JP,50,JP,S
4,2022,SE,FT,Lead Machine Learning Engineer,7500000,INR,95386,IN,50,IN,L


In [52]:
df2.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


In [53]:
df3.head()

,Job Title,Employment Type,Experience Level,Expertise Level,Salary,Salary Currency,Company Location,Salary in USD,Employee Residence,Company Size,Year
0,Data Engineer,Full-Time,Senior,Expert,210000,United States Dollar,United States,210000,United States,Medium,2023
1,Data Engineer,Full-Time,Senior,Expert,165000,United States Dollar,United States,165000,United States,Medium,2023
2,Data Engineer,Full-Time,Senior,Expert,185900,United States Dollar,United States,185900,United States,Medium,2023
3,Data Engineer,Full-Time,Senior,Expert,129300,United States Dollar,United States,129300,United States,Medium,2023
4,Data Scientist,Full-Time,Senior,Expert,140000,United States Dollar,United States,140000,United States,Medium,2023


## Creación de la variable `remote_ratio` para la concatenación de datasets

Antes de combinar los distintos datasets fue necesario asegurar que todos compartieran la misma estructura de columnas. Uno de los datasets originales no incluía la variable `remote_ratio`, lo que impedía realizar la concatenación directa.

Para resolver este problema, se creó la columna `remote_ratio` en los datasets donde no existía, inicializándola con valores nulos. De esta forma se pudo garantizar la compatibilidad entre las estructuras de los dataframes y realizar la concatenación correctamente.

La gestión de los valores faltantes de esta variable se abordará posteriormente durante la fase de limpieza de datos.

In [ ]:
# ---- LIMPIEZA BÁSICA DF3 (solo para igualar estructura) ----

df3 = df3.rename(columns={
    "Year": "work_year",
    "Job Title": "job_title",
    "Experience Level": "experience_level",
    "Employment Type": "employment_type",
    "Salary": "salary",
    "Salary Currency": "salary_currency",
    "Salary in USD": "salary_in_usd",
    "Employee Residence": "employee_residence",
    "Company Location": "company_location",
    "Company Size": "company_size"
})

# eliminar columna que no existe en los otros
df3 = df3.drop(columns=["Expertise Level"], errors="ignore")


# ---- HOMOGENEIZAR ESTRUCTURA ----

dfs = [df1, df2, df3]

for df in dfs:

    # crear columna remote_ratio si no existe
    if "remote_ratio" not in df.columns:
        df["remote_ratio"] = None

# ---- UNIR DATASETS ----

df_all = pd.concat(dfs, ignore_index=True)


print("Shape dataset combinado:", df_all.shape)
df_all.head()

Shape dataset combinado: (18745, 11)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2021,MI,FT,Data Scientist,30400000,CLP,40038,CL,100,CL,L
1,2021,MI,FT,BI Data Analyst,11000000,HUF,36259,HU,50,US,L
2,2020,MI,FT,Data Scientist,11000000,HUF,35735,HU,50,HU,L
3,2021,MI,FT,ML Engineer,8500000,JPY,77364,JP,50,JP,S
4,2022,SE,FT,Lead Machine Learning Engineer,7500000,INR,95386,IN,50,IN,L


In [55]:
def eda_basico(df):

    print("\n🔍 PRIMER VISTAZO A LOS DATOS")

    print("\n📌 Primeras filas")
    display(df.head())

    print("\n📌 Últimas filas")
    display(df.tail())

    print("\n📌 Muestra aleatoria")
    display(df.sample(5))


    print("\n🧱 ESTRUCTURA DEL DATAFRAME")

    print(f"\n📐 Dimensiones: {df.shape}")

    print("\n📋 Columnas")
    display(df.columns)

    print("\n🧠 Información general")
    df.info()

    print("\n📊 Tipos de datos")
    display(df.dtypes)


    print("\n📉 ESTADÍSTICAS DESCRIPTIVAS")

    print("\n📈 Variables numéricas")
    display(df.describe().T)

    print("\n🔤 Variables categóricas")
    display(df.describe(include='O').T)


    print("\n🔢 CARDINALIDAD")

    print("\nValores únicos por columna")
    display(df.nunique())


    print("\n🚫 VALORES NULOS")

    print("\nConteo de nulos")
    display(df.isnull().sum())

    print("\nPorcentaje de nulos (%)")
    display(((df.isnull().sum()/len(df))*100).round(2))


    print("\n📎 DUPLICADOS")

    dup = df.duplicated().sum()
    print(f"\nDuplicados: {dup}")

    if dup > 0:
        display(df[df.duplicated()])


    print("\n📊 VALUE COUNTS (categóricas)")

    cat_cols = df.select_dtypes(include="object").columns

    for col in cat_cols:
        print(f"\n📌 {col}")
        display(df[col].value_counts())


    # =========================
    # EXTRA PARA TU DATASET
    # =========================

    print("\n💰 SALARIO MEDIO POR JOB TITLE")

    display(
        df.groupby("job_title")["salary_in_usd"]
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )


    print("\n🚨 DETECCIÓN DE OUTLIERS EN SALARIO (IQR)")

    Q1 = df["salary_in_usd"].quantile(0.25)
    Q3 = df["salary_in_usd"].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df["salary_in_usd"] < lower) | (df["salary_in_usd"] > upper)]

    print(f"Outliers detectados: {len(outliers)}")

    display(outliers.head())


In [56]:
eda_basico(df_all)


🔍 PRIMER VISTAZO A LOS DATOS

📌 Primeras filas


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2021,MI,FT,Data Scientist,30400000,CLP,40038,CL,100,CL,L
1,2021,MI,FT,BI Data Analyst,11000000,HUF,36259,HU,50,US,L
2,2020,MI,FT,Data Scientist,11000000,HUF,35735,HU,50,HU,L
3,2021,MI,FT,ML Engineer,8500000,JPY,77364,JP,50,JP,S
4,2022,SE,FT,Lead Machine Learning Engineer,7500000,INR,95386,IN,50,IN,L



📌 Últimas filas


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
18740,2020,Senior,Full-Time,Data Scientist,412000,United States Dollar,412000,United States,None,United States,Large
18741,2021,Mid,Full-Time,Principal Data Scientist,151000,United States Dollar,151000,United States,None,United States,Large
18742,2020,Entry,Full-Time,Data Scientist,105000,United States Dollar,105000,United States,None,United States,Small
18743,2020,Entry,Contract,Business Data Analyst,100000,United States Dollar,100000,United States,None,United States,Large
18744,2021,Senior,Full-Time,Data Science Manager,7000000,Indian Rupee,94665,India,None,India,Large



📌 Muestra aleatoria


...

company_size
M         14000
Medium     2707
L          1181
Large       442
S           264
Small       151
Name: count, dtype: int64


💰 SALARIO MEDIO POR JOB TITLE


job_title
Analytics Engineering Manager     399880.000000
Data Science Tech Lead            375000.000000
Managing Director Data Science    286666.666667
AWS Data Architect                258000.000000
AI Architect                      252935.062500
Head of Machine Learning          250406.333333
Cloud Data Architect              250000.000000
Principal Data Engineer           230846.625000
Director of Data Science          213073.050847
Data Analytics Lead               209326.952381
Prompt Engineer                   205093.588235
Head of Data                      200192.768116
Principal Data Scientist          199749.576923
Data Infrastructure Engineer      198935.000000
Robotics Software Engineer        196625.000000
Name: salary_in_usd, dtype: float64


🚨 DETECCIÓN DE OUTLIERS EN SALARIO (IQR)
Outliers detectados: 332


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
46,2023,SE,FT,AI Scientist,1500000,ILS,417937,IL,0,IL,L
66,2024,MI,FT,AI Architect,800000,USD,800000,CA,100,CA,M
68,2024,EN,FT,Data Analyst,774000,USD,774000,MX,0,MX,M
69,2024,SE,FT,Analytics Engineer,750000,USD,750000,US,0,US,M
70,2024,MI,FT,Machine Learning Scientist,750000,USD,750000,US,0,US,M


## Resumen del Análisis Exploratorio (EDA)

Tras combinar los tres datasets originales en un único dataframe, se realizó un análisis exploratorio con el objetivo de comprender la estructura de los datos, detectar inconsistencias y definir las transformaciones necesarias antes de la fase de limpieza.

Durante la exploración se identificaron varios aspectos relevantes. En primer lugar, la variable `job_title` presenta una alta cardinalidad, con más de 150 títulos distintos que en muchos casos representan variaciones del mismo tipo de rol. Para facilitar el análisis posterior, será necesario agrupar estos títulos en categorías profesionales más amplias.

También se detectaron inconsistencias en algunas variables categóricas como `experience_level`, `employment_type` y `company_size`, donde los valores aparecían representados de distintas formas según el dataset de origen. Estas variables deberán normalizarse para asegurar coherencia en el análisis.

En las variables geográficas (`company_location` y `employee_residence`) se observaron diferencias en el formato de los países, apareciendo tanto códigos ISO como nombres completos. Estos valores se estandarizarán al formato de códigos ISO para evitar duplicidades en los análisis por país.

Además, se identificaron valores faltantes en la variable `remote_ratio`, que posteriormente deberán gestionarse para facilitar su interpretación.

Por último, al combinar los distintos datasets se detectaron **6.216 registros duplicados**, que deberán eliminarse durante la fase de limpieza para evitar distorsiones en los análisis posteriores.

Estas observaciones guían el proceso de limpieza, cuyo objetivo será obtener un dataset consistente, homogéneo y preparado para su posterior análisis y visualización en Power BI.